# Imports

In [ ]:
from datetime import datetime, timezone, timedelta
import html
import requests
from bs4 import BeautifulSoup
from collections import Counter
import json
import re

In [ ]:
def clean_html_to_text(raw_html: str) -> str:
    """Decodes HTML entities and strips tags into clean, human-readable text."""
    if not raw_html:
        return ""
    unescaped = html.unescape(raw_html)
    soup = BeautifulSoup(unescaped, "html.parser")
    return soup.get_text(separator="\n", strip=True)

def format_relative_time(dt: datetime) -> str:
    """Returns a readable relative time string like '3 hours ago' or '2 days ago'."""
    now = datetime.now(timezone.utc)
    diff = now - dt

    seconds = int(diff.total_seconds())
    if seconds < 60:
        return "just now"
    minutes = seconds // 60
    if minutes < 60:
        return f"{minutes} min ago"
    hours = minutes // 60
    if hours < 24:
        return f"{hours} hour{'s' if hours > 1 else ''} ago"
    days = hours // 24
    return f"{days} day{'s' if days > 1 else ''} ago"

# Greenhouse

In [7]:
ISRAEL_KEYWORDS = {
    # Country names & abbreviations
    "israel", "ישראל", "il",

    # Major Tech Hubs & Sub-districts
    "tel aviv", "tel-aviv", "tlv",
    "herzliya", "herzelia", "herzliya pituach", "herzliya pituach",
    "ramat gan", "ramat-gan",
    "givatayim", "giv'atayim",
    "holon", "bat yam", "bat-yam",
    "rishon lezion", "rishon le-zion", "rishon le zion",
    "petah tikva", "petach tikva", "petah-tikva", "petach-tikva",
    "bnei brak", "bene beraq",

    # Sharon & Central Plains
    "ra'anana", "raanana", "kfar saba", "kfar-saba", "hod hasharon",
    "netanya", "kfar netter", "shefayim", "yakum", "glil yam",
    "kfar yona", "kadima", "even yehuda",

    # Jerusalem Area
    "jerusalem", "beit shemesh", "modi'in", "modiin", "maccabim", "reut",

    # Rehovot / Shfela Science Parks
    "rehovot", "nes ziona", "ness ziona", "lod", "ramla", "ramlah",
    "kiryat gat", "ashdod", "ashkelon", "yavne",

    # North & Carmel Tech Parks
    "haifa", "matam", "yokneam", "yoqneam", "yokne'am",
    "caesarea", "hadera", "zichron yaakov", "zikhron ya'akov",
    "tirat carmel", "tirat karmel", "misgav", "migdal haemek", "migdal ha'emek",
    "afula", "carmiel", "karmiel", "nahariya", "akko", "acre",
    "nazareth", "tiberias",

    # South / Negev Cyber & Tech
    "beer sheva", "beersheba", "beer-sheva", "be'er sheva",
    "omer", "dimona", "sderot", "eilat",

    # Common Tech Park / Regional Designations
    "gush dan", "sharon", "merkaz", "galil", "negev",
    "shfela", "arava"
}

In [ ]:
def is_israel_location(location_name: str) -> bool:
    if not location_name:
        return False
    loc_lower = location_name.lower()
    return any(keyword in loc_lower for keyword in ISRAEL_KEYWORDS)

def get_text_preview(text: str, max_chars: int = 250) -> str:
    """Collapses whitespace/newlines and creates a clean snippet preview."""
    if not text:
        return "No description provided."
    # Collapse multiple consecutive newlines and spaces for display
    clean_snippet = " ".join(text.split())
    if len(clean_snippet) > max_chars:
        return clean_snippet[:max_chars].rstrip() + "..."
    return clean_snippet

def fetch_greenhouse_jobs(
    board_tokens: list[str],
    timeframe: str = "week",  # 'today' or 'week'
    filter_israel: bool = True,
    include_full_text: bool = True,
    preview_length: int = 200
) -> list[dict]:
    now = datetime.now(timezone.utc)
    delta = timedelta(days=1) if timeframe == "today" else timedelta(days=7)
    cutoff_time = now - delta

    results = []

    for token in board_tokens:
        url = f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs"
        params = {"content": "true"} if include_full_text else {}

        try:
            response = requests.get(url, params=params, timeout=20)
            if response.status_code == 404:
                print(f"[{token}] Board not found.")
                continue
            response.raise_for_status()

            data = response.json()
            jobs = data.get("jobs", [])

            for job in jobs:
                raw_updated_at = job.get("updated_at")
                if not raw_updated_at:
                    continue

                updated_at_dt = datetime.fromisoformat(raw_updated_at)
                if updated_at_dt < cutoff_time:
                    continue

                location_name = (job.get("location") or {}).get("name", "")
                if filter_israel and not is_israel_location(location_name):
                    continue

                raw_content = job.get("content", "")
                clean_description = clean_html_to_text(raw_content) if include_full_text else ""
                preview = get_text_preview(clean_description, max_chars=preview_length)

                results.append({
                    "id": job.get("id"),
                    "company": token,
                    "title": job.get("title"),
                    "location": location_name,
                    "updated_at_formatted": updated_at_dt.strftime("%Y-%m-%d %H:%M UTC"),
                    "time_ago": format_relative_time(updated_at_dt),
                    "url": job.get("absolute_url"),
                    "description_preview": preview,
                    "content_full": clean_description
                })

        except requests.RequestException as e:
            print(f"Error fetching board '{token}': {e}")

    return results

In [38]:
companies = [
    "appsflyer",
    "wizinc",
    "island",
    "gongio",
    "similarweb",
    "taboola",
    "riskified",
    "forter",
    "pagayais",
    "datadog",
    "cloudflare"
]

In [39]:
# include_full_text=True is needed to download job descriptions
jobs = fetch_greenhouse_jobs(
    board_tokens=companies,
    timeframe="today",
    filter_israel=True,
    include_full_text=True,
    preview_length=200
)

company_counts = Counter(job["company"] for job in jobs)

print(f"\nFound {len(jobs)} jobs updated in the past week across {len(company_counts)} companies:\n")

# Display in descending order of job count
for company, count in company_counts.most_common():
    print(f"{company}: {count} job{'s' if count > 1 else ''}")


Found 3 jobs updated in the past week across 2 companies:

taboola: 2 jobs
appsflyer: 1 job


In [42]:
target_company = "taboola"

filtered_jobs = [
    job for job in jobs 
    if not target_company or job["company"].lower() == target_company.lower()
]

if not filtered_jobs:
    print(f"No jobs found for company: '{target_company}'")
else:
    for job in filtered_jobs:
        print(f"[{job['company'].upper()}] {job['title']}")
        print(f"  ID:          {job['id']}")
        print(f"  Location:    {job['location']}")
        print(f"  Posted/Edit: {job['updated_at_formatted']} ({job['time_ago']})")
        print(f"  Apply URL:   {job['url']}")
        print(f"  Preview:     {job['description_preview']}")
        print("-" * 70)

[TABOOLA] Advertising Sales Manager - (Visa and Relocation Provided)
  ID:          8354774
  Location:    Bangkok, Thailand
  Posted/Edit: 2026-11-05 02:19 UTC (8 hours ago)
  Apply URL:   https://www.taboola.com/careers/job/8354774?gh_jid=8354774
  Preview:     Realize your potential by joining the leading performance-driven advertising company! As an Advertising Sales Manager on the Enterprise Team, you’ll play a vital role in focusing to identify, engage a...
----------------------------------------------------------------------
[TABOOLA] Software Engineer Intern
  ID:          8229871
  Location:    Tel Aviv, Israel
  Posted/Edit: 2026-11-04 12:00 UTC (22 hours ago)
  Apply URL:   https://www.taboola.com/careers/job/8229871?gh_jid=8229871
  Preview:     Realize your potential by joining the leading performance-driven advertising company! As a Software Engineer Intern on the Ad Creatives team in our Tel Aviv office, you’ll work on large-scale systems...
----------------------------

# Comeet

In [ ]:
POSITIONS_RE = re.compile(r"\bCOMPANY_POSITIONS_DATA\s*=\s*")

def fetch_comeet_jobs(slug: str, company_id: str, location_filter: str | None = None, timeframe: str | None = None) -> list:
    r = requests.get(f"https://www.comeet.com/jobs/{slug}/{company_id}", timeout=30)
    r.raise_for_status()
    page_html = r.text
    
    if not re.search(r"\bCOMPANY_DATA\s*=", page_html):
        raise ValueError(f"No company data found. Check slug '{slug}' and ID '{company_id}'.")

    match = POSITIONS_RE.search(page_html)
    if not match:
        raise ValueError("COMPANY_POSITIONS_DATA not found: the page layout may have changed.")
    
    positions, _ = json.JSONDecoder().raw_decode(page_html, match.end())

    groups = {}
    for p in positions:
        groups.setdefault(p["uid"].split("-")[0], []).append(p)

    # Calculate time cutoff
    now = datetime.now(timezone.utc)
    cutoff_time = None
    if timeframe == "today":
        cutoff_time = now - timedelta(days=1)
    elif timeframe == "week":
        cutoff_time = now - timedelta(days=7)

    jobs = []
    for base, rows in groups.items():
        primary = next((p for p in rows if p["uid"] == base), rows[0])
        
        # --- Time Parsing & Filtering ---
        raw_updated = primary.get("time_updated")
        updated_dt = None
        
        if raw_updated:
            # Replace 'Z' with '+00:00' to ensure fromisoformat works across Python versions
            clean_updated = raw_updated.replace("Z", "+00:00")
            try:
                updated_dt = datetime.fromisoformat(clean_updated)
            except ValueError:
                pass
        
        # Skip if the job is older than our cutoff time
        if cutoff_time and updated_dt and updated_dt < cutoff_time:
            continue
        
        # Build locations list
        locations = []
        israel_match = False
        for p in rows:
            loc = p.get("location") or {}
            place = loc.get("city") or loc.get("name")
            country = loc.get("country") or ""
            if place and place.lower() != "remote":
                locations.append({"place": place, "country": country})
                
            # Check against our location filter
            if location_filter:
                if location_filter.lower() in country.lower():
                    israel_match = True

        # Skip if we are filtering for Israel and this job isn't there
        if location_filter and not israel_match:
            continue

        # Extract full description
        sections = sorted(primary.get("custom_fields", {}).get("details") or [], key=lambda d: d.get("order", 0))
        html_desc = "".join(
            (f"<h3>{s['name']}</h3>" if s.get("name") else "") + s["value"]
            for s in sections if (s.get("value") or "").strip()
        )

        time_ago_str = format_relative_time(updated_dt) if updated_dt else "Unknown"

        jobs.append({
            "company": slug,
            "id": base,
            "title": primary.get("name"),
            "time_ago": time_ago_str,
            "updated_at_formatted": updated_dt.strftime("%Y-%m-%d %H:%M UTC") if updated_dt else raw_updated,
            "workplace_type": primary.get("workplace_type"),
            "employment_type": primary.get("employment_type"),
            "experience_level": primary.get("experience_level"),
            "locations": locations,
            "url": primary.get("url_comeet_hosted_page") or primary.get("url_active_page"),
            "description_preview": clean_html_to_text(html_desc)[:200] + "..." if html_desc else "No description",
        })
        
    return jobs

In [48]:
# List of tuples containing (slug, company_id)
comeet_companies = [
    ("Claroty", "F2.004"),
    ("sentra", "87.00B"),
    ("gett", "A0.002"),
    ("global-e", "62.002"),
]

for slug, company_id in comeet_companies:
    print(f"--- Searching Comeet positions for: {slug} ---")
    try:
        # Added timeframe="week"
        company_jobs = fetch_comeet_jobs(slug=slug, company_id=company_id, location_filter="il", timeframe="today")
        print(f"Found {len(company_jobs)} positions in Israel:\n")
        
        for job in company_jobs:
            loc_str = " | ".join([f"{l.get('place', 'Unknown')}, {l.get('country', 'Unknown')}" for l in job['locations']])
            
            print(f"- {job['title']} (ID: {job['id']})")
            print(f"  Location: {loc_str}")
            print(f"  Updated: {job['time_ago']} ({job['updated_at_formatted']})")
            #print(f"  Employment Type: {job['employment_type']}")
            #print(f"  URL: {job['url']}")
            #print(f"  Preview: {job['description_preview']}\n")
            
    except Exception as e:
        print(f"Error fetching {slug}: {e}\n")

--- Searching Comeet positions for: Claroty ---
Found 12 positions in Israel:

- AI Platform, Backend Software Engineer (ID: 66.271)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- AI Platform- Software Engineer Team Leader (ID: CE.079)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- Backend Platform Engineer (ID: 58.479)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- Backend Software Engineer (ID: 0A.374)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- Data Platform Engineer (ID: CA.471)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- DevOps Engineer (ID: BB.478)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- LLM Engineer (ID: 86.473)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- Paralegal (ID: 36.371)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- QA Automation Engineer (ID: 3C.374)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- Security Data Analyst (ID: EA.471)
  Location: Tel Aviv, IL
  Updated: Unknown (None)
- Senior Security Op